# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and analyzing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print("Dataset Loaded!")
print("Name:", metadata.name)
print("Description:", metadata.description)
print("Identifier:", getattr(metadata, "identifier", None))
print("License:", metadata.license)
print("Authors (@id):")
for author in getattr(metadata, "author", []):
    print("  -", author['@id'])

## 2. Data Overview
Review available record sets, fields, and their `@id` values.

**Note:** The FAIR² dataset contains multiple tables (record sets) describing clinical features, hormone profiles, and genetic variants. These tables are described in the Croissant metadata under the `recordSet` property.

Let's list all record sets and their fields, referencing each by their `@id`.

In [ ]:
# List the record sets and fields (all referenced by @id)
record_sets = getattr(metadata, 'recordSet', [])
print("Record Sets found:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']} | Name: {rs.get('name', 'N/A')}")
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print("  Fields:")
    for f in fields:
        print(f"    - Field @id: {f['@id']} | Name: {f.get('name', 'N/A')}")
    print()

# Display a sample record from each record set
for rs in record_sets:
    print(f"Sample from RecordSet @id {rs['@id']}:")
    recs = list(dataset.records(record_set=rs['@id']))
    if recs:
        print(recs[0])
    else:
        print('No records found.')
    print()

## 3. Data Extraction
Load data from each record set into DataFrames for analysis. All object references use their `@id` values.

We extract tables for:
- Table 1: Clinical features
- Table 2: Hormone/imaging/ultrasound results
- Table 3: Genetic variants

Below, the extraction references each record set by its `@id`.

In [ ]:
# Extract data from each record set
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Display columns and head for the first available record set
if record_set_ids:
    first_id = record_set_ids[0]
    print(f"Columns in RecordSet @{first_id}: {dataframes[first_id].columns.tolist()}")
    display(dataframes[first_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filter, normalize, and group data.

We'll select a numeric field (e.g., age at diagnosis) and a categorical field (e.g., presence of clinical feature) from Table 1 for illustration.

**Reference all fields and record sets by their `@id` throughout.**

In [ ]:
# Choose a record set for analysis (Table 1 if available)
if record_set_ids:
    rs_id = record_set_ids[0]  # assume first record set is Table 1
    df = dataframes[rs_id]

    # Find a numeric field (search by @id or known fields)
    numeric_field_id = None
    group_field_id = None
    fields = record_sets[0].get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    for field in fields:
        # Example: field name 'Age at diagnosis'
        if 'age' in field.get('name', '').lower() or field.get('dataType', '') == 'schema:Integer' or field.get('dataType', '') == 'schema:Float':
            numeric_field_id = field['@id']
        # Example: field name 'clinical_feature' for grouping
        if 'clitomegaly' in field.get('name', '').lower():
            group_field_id = field['@id']

    # If found, perform filtering and normalization
    if numeric_field_id and numeric_field_id in df.columns:
        threshold = 10  # Example threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a key categorical field (e.g., clitomegaly)
        if group_field_id and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id).mean()
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print('Numeric field not found for EDA.')
else:
    print('No record sets available for EDA.')

## 5. Visualization
Visualize data distributions and relationships using fields referenced by their `@id`.

We plot the age distribution and compare normalization: all fields referenced by their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric distributions if available
if record_set_ids and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If normalized exists (from EDA), show side-by-side
    norm_col = f"{numeric_field_id}_normalized"
    if norm_col in filtered_df.columns:
        fig, axes = plt.subplots(1, 2, figsize=(12,4))
        sns.histplot(df[numeric_field_id], ax=axes[0], kde=True)
        axes[0].set_title(f"Original {numeric_field_id}")
        sns.histplot(filtered_df[norm_col], ax=axes[1], kde=True)
        axes[1].set_title(f"Normalized {numeric_field_id}")
        plt.show()
else:
    print('No data available for visualization.')

## 6. Conclusion
This notebook demonstrated how to load, explore, and process the FAIR² dataset using `mlcroissant`. Key steps included referencing all entities by their `@id`, loading multiple record sets, performing basic filtering and normalization, grouping data, and visualizing numeric distributions.

Due to the small sample size (N=3) and limited cohort scope, results should be interpreted as illustrative only. For further research or clinical reproducibility, additional data is required.